# 🐍 Mamba Official (Mamba1 / Mamba2 / Mamba3) — Eye Image HB Estimation
**Repo:** [state-spaces/mamba](https://github.com/state-spaces/mamba)  
**Tasks:** ① Binary classification (Anemic vs Non-Anemic) ② HB regression (g/dL)  
**Data:** Retinal/conjunctival eye images + Excel sheet with Patient ID & HB values

---


In [ ]:
# ── 0. Clone & install (run once on Kaggle) ───────────────────────────────
import subprocess, sys

# Clone repo
subprocess.run(["git", "clone", "--depth=1",
                "https://github.com/state-spaces/mamba.git", "mamba_repo"],
               check=False)

# Install dependencies (Kaggle GPU)
packages = [
    "mamba-ssm",          # official compiled package (CUDA)
    "causal-conv1d",
    "einops",
    "timm",
    "pandas",
    "openpyxl",
    "scikit-learn",
    "matplotlib",
    "tqdm",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)

print("✅ Dependencies installed")


In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────
import os, sys, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error
import matplotlib.pyplot as plt
from tqdm import tqdm
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


In [ ]:
# ── 2. Config — EDIT THESE PATHS ─────────────────────────────────────────
CFG = dict(
    # ---- Data paths (Kaggle) ----
    image_dir   = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/left_eye",
    csv_path    = "/kaggle/input/datasets/junaidgpu/imagehb/dataset/dataset/merge_excel_1.xlsx",
    image_col   = "Patient ID",     # column with filename (no extension)
    hb_col      = "HB",             # column with HB value
    hb_thresh   = 12.0,             # < threshold → anemic (label=0)

    # ---- Model ----
    img_size    = 224,
    patch_size  = 16,
    embed_dim   = 256,
    depth       = 6,                # number of Mamba blocks
    mamba_ver   = "mamba1",         # "mamba1" | "mamba2" | "mamba3"

    # ---- Training ----
    epochs      = 30,
    batch_size  = 8,
    lr          = 1e-4,
    weight_decay= 1e-2,
    cls_weight  = 1.0,              # loss weight for classification
    reg_weight  = 0.5,              # loss weight for regression
    val_split   = 0.2,
    seed        = 42,
)


In [ ]:
# ── 3. Dataset ────────────────────────────────────────────────────────────
class EyeHBDataset(Dataset):
    def __init__(self, df, image_dir, image_col, hb_col, hb_thresh, transform=None):
        self.df        = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.image_col = image_col
        self.hb_col    = hb_col
        self.hb_thresh = hb_thresh
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        pid      = str(row[self.image_col])
        hb_val   = float(row[self.hb_col])
        label    = 0 if hb_val < self.hb_thresh else 1   # 0=anemic, 1=normal

        # Try common extensions
        img_path = None
        for ext in [".jpg", ".jpeg", ".png", ".bmp", ""]:
            p = os.path.join(self.image_dir, pid + ext)
            if os.path.exists(p):
                img_path = p
                break
        if img_path is None:
            raise FileNotFoundError(f"Image not found for patient {pid}")

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long), torch.tensor([[hb_val]], dtype=torch.float32)


def get_transforms(img_size, train=True):
    if train:
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])


In [ ]:
# ── 4. Load data & create loaders ────────────────────────────────────────
df = pd.read_excel(CFG["csv_path"])
print(f"Total samples: {len(df)}")
print(df[[CFG['image_col'], CFG['hb_col']]].head())

train_df, val_df = train_test_split(df, test_size=CFG["val_split"],
                                    random_state=CFG["seed"],
                                    stratify=(df[CFG["hb_col"]] < CFG["hb_thresh"]).astype(int))

train_ds = EyeHBDataset(train_df, CFG["image_dir"], CFG["image_col"],
                         CFG["hb_col"], CFG["hb_thresh"],
                         transform=get_transforms(CFG["img_size"], train=True))
val_ds   = EyeHBDataset(val_df,   CFG["image_dir"], CFG["image_col"],
                         CFG["hb_col"], CFG["hb_thresh"],
                         transform=get_transforms(CFG["img_size"], train=False))

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds)}  |  Val: {len(val_ds)}")
print(f"Anemic in train: {(train_df[CFG['hb_col']] < CFG['hb_thresh']).sum()}")


In [ ]:
# ── 5. Mamba-based Vision Model (Dual-Head) ──────────────────────────────
#
# Works in two modes:
#   (a) mamba_ssm installed   → uses compiled CUDA Mamba blocks
#   (b) mamba_ssm NOT installed → uses pure-PyTorch SSM fallback
#
from einops import rearrange, repeat
try:
    from mamba_ssm import Mamba as MambaLayer
    MAMBA_FAST = True
    print("✅ Using compiled mamba_ssm (fast CUDA path)")
except ImportError:
    MAMBA_FAST = False
    print("⚠️  mamba_ssm not found — using pure-PyTorch SSM (slower but works everywhere)")

# ------ Pure-PyTorch fallback SSM ----------------------------------------
class PureMambaLayer(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2):
        super().__init__()
        d_inner       = int(expand * d_model)
        self.in_proj  = nn.Linear(d_model, d_inner * 2, bias=False)
        self.conv1d   = nn.Conv1d(d_inner, d_inner, kernel_size=d_conv,
                                   padding=d_conv-1, groups=d_inner)
        self.act      = nn.SiLU()
        self.x_proj   = nn.Linear(d_inner, d_state*2+1, bias=False)
        self.dt_proj  = nn.Linear(1, d_inner, bias=True)
        self.out_proj = nn.Linear(d_inner, d_model, bias=False)
        A = repeat(torch.arange(1, d_state+1, dtype=torch.float32), 'n -> d n', d=d_inner)
        self.A_log    = nn.Parameter(torch.log(A))
        self.D        = nn.Parameter(torch.ones(d_inner))
        self.d_inner  = d_inner
        self.d_state  = d_state

    def forward(self, x):
        B, L, _ = x.shape
        xz  = self.in_proj(x)
        x_, z = xz.chunk(2, dim=-1)
        x_  = rearrange(x_, 'b l d -> b d l')
        x_  = self.conv1d(x_)[..., :L]
        x_  = rearrange(x_, 'b d l -> b l d')
        x_  = self.act(x_)
        bcd = self.x_proj(x_)
        B_  = bcd[..., :self.d_state]
        C   = bcd[..., self.d_state:2*self.d_state]
        dt  = F.softplus(self.dt_proj(bcd[..., -1:]))
        A   = -torch.exp(self.A_log.float())
        dA  = torch.exp(dt.unsqueeze(-1) * A)
        dB  = dt.unsqueeze(-1) * B_.unsqueeze(2)
        h   = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        ys  = []
        for i in range(L):
            h = dA[:, i] * h + dB[:, i] * x_[:, i:i+1].unsqueeze(-1)
            ys.append((h * C[:, i].unsqueeze(1)).sum(-1))
        y = torch.stack(ys, dim=1)
        y = y + x_ * self.D
        y = y * self.act(z)
        return self.out_proj(y)

# ------ Select Mamba layer based on version config ------------------------
def make_mamba_layer(d_model, d_state=16, d_conv=4, expand=2, ver="mamba1"):
    if MAMBA_FAST:
        if ver == "mamba1":
            from mamba_ssm import Mamba
            return Mamba(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        elif ver == "mamba2":
            from mamba_ssm.modules.mamba2_simple import Mamba2Simple
            return Mamba2Simple(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)
        elif ver == "mamba3":
            from mamba_ssm.modules.mamba3 import Mamba3
            return Mamba3(d_model=d_model)
    return PureMambaLayer(d_model, d_state, d_conv, expand)

# ------ Vision Mamba Backbone -------------------------------------------
from timm.models.layers import DropPath, to_2tuple, trunc_normal_

class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=256):
        super().__init__()
        img_size = to_2tuple(img_size); patch_size = to_2tuple(patch_size)
        self.num_patches = (img_size[0]//patch_size[0]) * (img_size[1]//patch_size[1])
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1,2)

class VisionMambaBlock(nn.Module):
    def __init__(self, d_model, d_state=16, d_conv=4, expand=2, drop_path=0., ver="mamba1"):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = make_mamba_layer(d_model, d_state, d_conv, expand, ver)
        self.drop_path = DropPath(drop_path) if drop_path > 0 else nn.Identity()
    def forward(self, x):
        return x + self.drop_path(self.mamba(self.norm(x)))

class VisionMambaDualHead(nn.Module):
    """Vision Mamba with classification + regression heads."""
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=2,
                 embed_dim=256, depth=6, d_state=16, d_conv=4, expand=2,
                 drop_path_rate=0.1, ver="mamba1"):
        super().__init__()
        self.patch_embed = PatchEmbed(img_size, patch_size, in_chans, embed_dim)
        N = self.patch_embed.num_patches
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, N+1, embed_dim))
        trunc_normal_(self.cls_token, std=0.02); trunc_normal_(self.pos_embed, std=0.02)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, depth)]
        self.blocks = nn.ModuleList([
            VisionMambaBlock(embed_dim, d_state, d_conv, expand, dpr[i], ver)
            for i in range(depth)])
        self.norm    = nn.LayerNorm(embed_dim)
        self.cls_head = nn.Linear(embed_dim, num_classes)
        self.reg_head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim//2), nn.GELU(), nn.Linear(embed_dim//2, 1))
        self.apply(self._iw)

    def _iw(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.zeros_(m.bias); nn.init.ones_(m.weight)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        x = torch.cat([self.cls_token.expand(B,-1,-1), x], dim=1) + self.pos_embed
        for blk in self.blocks: x = blk(x)
        feat = self.norm(x)[:, 0]
        return self.cls_head(feat), self.reg_head(feat)

model = VisionMambaDualHead(
    img_size=CFG["img_size"], patch_size=CFG["patch_size"],
    embed_dim=CFG["embed_dim"], depth=CFG["depth"], ver=CFG["mamba_ver"]
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: VisionMamba-{CFG['mamba_ver'].upper()}  |  Params: {total_params/1e6:.2f}M")


In [ ]:
# ── 6. Training setup ────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(model.parameters(),
                               lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["epochs"])
ce_loss   = nn.CrossEntropyLoss()
mse_loss  = nn.MSELoss()

def compute_loss(logits, hb_pred, labels, hb_true):
    cl = ce_loss(logits, labels)
    rl = mse_loss(hb_pred, hb_true)
    return CFG["cls_weight"] * cl + CFG["reg_weight"] * rl, cl.item(), rl.item()


In [ ]:
# ── 7. Train & Validate ───────────────────────────────────────────────────
history = {"train_loss":[], "val_loss":[], "val_acc":[], "val_mae":[]}

best_val_loss = float("inf")
for epoch in range(1, CFG["epochs"]+1):
    # — Train —
    model.train()
    train_loss = 0
    for imgs, labels, hb_true in tqdm(train_loader, desc=f"Ep {epoch}/{CFG['epochs']} [train]", leave=False):
        imgs, labels, hb_true = imgs.to(DEVICE), labels.to(DEVICE), hb_true.to(DEVICE)
        optimizer.zero_grad()
        logits, hb_pred = model(imgs)
        loss, _, _ = compute_loss(logits, hb_pred, labels, hb_true)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_loss += loss.item()
    scheduler.step()
    train_loss /= len(train_loader)

    # — Validate —
    model.eval()
    val_loss, correct, total = 0, 0, 0
    all_hb_pred, all_hb_true = [], []
    with torch.no_grad():
        for imgs, labels, hb_true in val_loader:
            imgs, labels, hb_true = imgs.to(DEVICE), labels.to(DEVICE), hb_true.to(DEVICE)
            logits, hb_pred = model(imgs)
            loss, _, _ = compute_loss(logits, hb_pred, labels, hb_true)
            val_loss += loss.item()
            preds     = logits.argmax(1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)
            all_hb_pred.extend(hb_pred.cpu().squeeze().tolist())
            all_hb_true.extend(hb_true.cpu().squeeze().tolist())
    val_loss /= len(val_loader)
    val_acc   = correct / total
    val_mae   = mean_absolute_error(all_hb_true, all_hb_pred)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_mae"].append(val_mae)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_mamba_model.pth")

    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:3d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} "
              f"| Val Acc: {val_acc:.3f} | Val MAE (HB): {val_mae:.2f} g/dL")


In [ ]:
# ── 8. Plot training curves ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_r = range(1, len(history["train_loss"])+1)

axes[0].plot(epochs_r, history["train_loss"], label="Train")
axes[0].plot(epochs_r, history["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(epochs_r, history["val_acc"], color="green")
axes[1].set_title("Val Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylim(0,1)

axes[2].plot(epochs_r, history["val_mae"], color="orange")
axes[2].set_title("Val MAE — HB (g/dL)"); axes[2].set_xlabel("Epoch")

plt.tight_layout(); plt.savefig("training_curves.png", dpi=120); plt.show()
print("Saved: training_curves.png")


In [ ]:
# ── 9. Final evaluation on validation set ────────────────────────────────
model.load_state_dict(torch.load("best_mamba_model.pth", map_location=DEVICE))
model.eval()

all_preds, all_labels, all_hb_pred, all_hb_true = [], [], [], []
with torch.no_grad():
    for imgs, labels, hb_true in val_loader:
        imgs = imgs.to(DEVICE)
        logits, hb_pred = model(imgs)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.tolist())
        all_hb_pred.extend(hb_pred.cpu().squeeze().tolist())
        all_hb_true.extend(hb_true.cpu().squeeze().tolist())

print("Classification Report:")
print(classification_report(all_labels, all_preds, target_names=["Anemic","Normal"]))
print(f"HB Regression  MAE : {mean_absolute_error(all_hb_true, all_hb_pred):.4f} g/dL")
print(f"HB Regression RMSE : {np.sqrt(np.mean((np.array(all_hb_true)-np.array(all_hb_pred))**2)):.4f} g/dL")

# Scatter: true vs predicted HB
plt.figure(figsize=(6,6))
plt.scatter(all_hb_true, all_hb_pred, alpha=0.5)
mn, mx = min(all_hb_true), max(all_hb_true)
plt.plot([mn,mx],[mn,mx],'r--', label="Perfect")
plt.xlabel("True HB (g/dL)"); plt.ylabel("Predicted HB (g/dL)")
plt.title("HB Regression: True vs Predicted"); plt.legend()
plt.tight_layout(); plt.savefig("hb_scatter.png", dpi=120); plt.show()
